# Improving Cross-Dataset Generalization in Deepfake Detection using Hybrid Spatial-Frequency Transfer Learning

**Goal:** Explicitly measure, analyze, and reduce the domain shift (generalization gap) between datasets (e.g., Train on FF++ → Test on Celeb-DF). Not just focused on accuracy, but on how well the model generalizes across domains.

### Dataset Overview
*   **Originals (Real):** 300 videos from YouTube
*   **Manipulated (Fake):** 900 videos total (300 Deepfakes, 300 FaceSwap, 300 Face2Face)

### The Core Contribution
We define a concrete evaluation metric:
`Generalization Gap = Accuracy (FF++) - Accuracy (Celeb-DF)`

We compare three architectural approaches:
1. **RGB Only:** Expected to have a *large drop* (learns dataset artifacts).
2. **FFT Only:** Expected to have a *moderate drop* (more domain-agnostic, captures frequency inconsistencies).
3. **Hybrid (RGB + FFT):** Our proposed approach, expected to have the *smallest drop* and best generalization. Plus, we test robust data augmentations to further force transfer learning.

## 1. Imports & Preprocessing (RGB + FFT)

In [ ]:
import os
import datetime
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

from fakers import *

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


### Extracting FFT Features
The FFT captures frequency inconsistencies that are missed by the spatial RGB domain.

In [2]:
def extract_fft(image):
    """
    Computes the Fast Fourier Transform of an RGB image.
    Returns a 3-channel magnitude spectrum image.
    """
    # Convert to grayscale for frequency analysis
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Perform 2D FFT
    f = np.fft.rfft2(gray)
    fshift = np.fft.fftshift(f)
    
    # Calculate magnitude spectrum (log scale for visualization/training stability)
    magnitude_spectrum = 20 * np.log(np.abs(fshift) + 1e-8)
    
    # Normalize to 0-255
    magnitude_spectrum = cv2.normalize(magnitude_spectrum, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    
    # Convert back to 3 channels so it can be passed through standard CNNs like EfficientNet
    fft_3channel = cv2.cvtColor(magnitude_spectrum, cv2.COLOR_GRAY2BGR)
    return fft_3channel

## 2. Dataset & Data Augmentation Pipelines
Applying blur, compression, and noise helps reduce reliance on visual shortcuts (Experiment 2 & 4).

In [2]:
# Base transforms for baseline and standard extraction
base_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Heavy augmentation for Experiments to reduce "shortcut" learning
aug_transform = A.Compose([
    A.Resize(224, 224),
    A.ImageCompression(quality_range = (5, 20), p=0.5), # Compression
    A.GaussianBlur(blur_limit=(1, 3), p=0.5),           # Blur
    A.GaussNoise(std_range=(0.05, 0.1), p=0.5),         # Noise
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

## 3. Two-Branch Model Architecture
RGB -> CNN -> Features
FFT -> CNN -> Features
Concat -> Classifier -> Real/Fake

In [3]:
from fakers import HybridDeepfakeDetector

## 4. Evaluation & Metrics Loop (Step 7)

In [4]:
from fakers import evaluate_generalization_gap, evaluate_model

## 5. Running the Experiments (Step 6)

In [5]:
def train_model(model, training_loader, validation_loader, epochs, device="cpu"):

    trg_losses, val_losses, accs, f1s, aucs = [], [], [], [], []
    loss_fn = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters())

    best_vf1 = -1_000_000.

    for epoch_number in range(epochs):
        print('EPOCH {}:'.format(epoch_number + 1))

        # Make sure gradient tracking is on, and do a pass over the data
        model.train(True)

        running_loss = 0.
        last_loss = 0.
        
        for i, data in enumerate(training_loader):
            
            # get data
            rgb_imgs, fft_imgs, labels = data
            rgb_imgs, fft_imgs, labels = rgb_imgs.to(device), fft_imgs.to(device), labels.to(device)

            # predict and backpropagate
            optimizer.zero_grad()
            outputs = model(rgb_imgs, fft_imgs)

            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            # report every hundred batches
            running_loss += loss.item()
            if (i + 1) % 50 == 0:
                last_loss = running_loss / 50 # loss per batch
                print('  batch {} loss: {}'.format(i + 1, last_loss))
                running_loss = 0.

        # evaluate each epoch
        val_loss, val_acc, val_f1, val_auc = evaluate_model(
            model=model,
            dataloader=validation_loader,
            device=device
        )
        print('LOSS train {} valid {}'.format(last_loss, val_loss))
        print(
            f"val acc: {val_acc:.4f} | "
            f"val f1: {val_f1:.4f} | "
            f"val auc: {val_auc:.4f}"
        )
        trg_losses.append(last_loss)
        val_losses.append(val_loss)
        accs.append(val_acc)
        f1s.append(val_f1)
        aucs.append(val_auc)

        # save best model by f1
        if best_vf1 < val_f1:
            best_vf1 = val_f1
            timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            model_path = '../models/model_{}_{}_{}'.format(model.mode, timestamp, epoch_number)
            torch.save(model.state_dict(), model_path)
    
    return trg_losses, val_losses, accs, f1s, aucs

### Face only tests

In [6]:
gen = torch.Generator().manual_seed(42)

# fpp image datasets
fpp_frames_path = "../data/fpp-face/" # change to appropriate directory
fpp_real_path = fpp_frames_path + "real/"
fpp_fake_path = fpp_frames_path + "fake/"

fpp_img_paths = [fpp_real_path + img_path for img_path in os.listdir(fpp_real_path)] + [fpp_fake_path + img_path for img_path in os.listdir(fpp_fake_path)]
fpp_img_labels = [0] * len(os.listdir(fpp_real_path)) + [1] * len(os.listdir(fpp_fake_path))

# celeb-df dataset
cdf_frames_path = "../data/celebdf-face/" # change to appropriate directory. only using test set here.
cdf_real_path = cdf_frames_path + "real/"
cdf_fake_path = cdf_frames_path + "fake/"

cdf_img_paths = [cdf_real_path + img_path for img_path in os.listdir(cdf_real_path)] + [cdf_fake_path + img_path for img_path in os.listdir(cdf_fake_path)]
cdf_img_labels = [0] * len(os.listdir(cdf_real_path)) + [1] * len(os.listdir(cdf_fake_path))

# datasets and loaders with augmentation
fpp_wtrf_train_set, fpp_wtrf_val_set = random_split(DeepfakeImageDataset(fpp_img_paths, fpp_img_labels, transform=aug_transform), [0.8, 0.2], gen)
fpp_wtrf_train_loader = DataLoader(fpp_wtrf_train_set, 64, shuffle=True)
fpp_wtrf_val_loader = DataLoader(fpp_wtrf_val_set, 64, shuffle=True)

# datasets and loaders without augmentation
fpp_train_set, fpp_val_set = random_split(DeepfakeImageDataset(fpp_img_paths, fpp_img_labels, transform=base_transform), [0.8, 0.2], gen)
fpp_train_loader = DataLoader(fpp_train_set, 64, shuffle=True)
fpp_val_loader = DataLoader(fpp_val_set, 64, shuffle=True)

# celeb-df validation dataset
cdf_dataset = DeepfakeImageDataset(cdf_img_paths, cdf_img_labels, transform=base_transform)
cdf_dataloader = DataLoader(cdf_dataset, 64, shuffle=True)

In [ ]:
# Placeholder loop explicitly executing the 4 major components of your thesis

# Let's assume we have our data loaders for FF++ and Celeb-DF ready
# ffpp_loader = DataLoader(DeepfakeDataset(ffpp_paths, labels, 'hybrid'), batch_size=32)
# celeb_df_loader = DataLoader(DeepfakeDataset(celebdf_paths, labels, 'hybrid'), ... )

# Keep track of experiment results to print standard tables later
experiment_results = {}

# ---------------------------------------------------------
# 1. Experiment 1 - Baseline (RGB only, No Augmentation)
# Expected Result: Massive domain shift drop (large gap)
# ---------------------------------------------------------

model_rgb = HybridDeepfakeDetector(mode='rgb').to(device)
stats = train_model(model_rgb, fpp_train_loader, fpp_val_loader, 10)
gap_rgb = evaluate_generalization_gap(model_rgb, fpp_val_loader, cdf_dataloader, 'rgb')
experiment_results['RGB Only'] = gap_rgb, stats

# ---------------------------------------------------------
# 2. Experiment 2 - Augmentation (RGB only)
# Expected Result: Small improvement to generalization gap over pure RGB
# ---------------------------------------------------------

model_aug = HybridDeepfakeDetector(mode='rgb').to(device)
stats = train_model(model_aug, fpp_wtrf_train_loader, fpp_wtrf_val_loader, 10)
gap_aug = evaluate_generalization_gap(model_aug, fpp_wtrf_val_loader, cdf_dataloader, 'rgb')
experiment_results['RGB + Augmentation'] = gap_aug, stats

# ---------------------------------------------------------
# 3. Experiment 3 - FFT Only (Frequency structure)
# Expected Result: Generalization gap is moderate compared to RGB only
# ---------------------------------------------------------

model_fft = HybridDeepfakeDetector(mode='fft').to(device)
stats = train_model(model_fft, fpp_train_loader, fpp_val_loader, 10)
gap_fft = evaluate_generalization_gap(model_fft, fpp_val_loader, cdf_dataloader, 'fft')
experiment_results['FFT Only'] = gap_fft, stats

# ---------------------------------------------------------
# 4. Experiment 4 - Hybrid/Combined (RGB + FFT + Augmentation)
# Expected Result: Smallest Generalization drop due to robust frequency/spatial combos
# ---------------------------------------------------------

model_hyb = HybridDeepfakeDetector(mode='hybrid').to(device)
stats = train_model(model_hyb, fpp_wtrf_train_loader, fpp_wtrf_val_loader, 10)
gap_hybrid = evaluate_generalization_gap(model_hyb, fpp_wtrf_val_loader, cdf_dataloader, 'hybrid')
experiment_results['Hybrid (RGB + FFT)'] = gap_hybrid, stats

# Visualizing the contribution
# print("### Domain Shift Contributions ###")
# for exp_name, gap in experiment_results.items():
#     print(f"{exp_name:25s} | Gap: {gap*100:.2f}%")

### Whole frame tests

In [8]:
gen = torch.Generator().manual_seed(42)

# fpp image datasets
fpp_frames_path = "../data/fpp-frame/" # change to appropriate directory
fpp_real_path = fpp_frames_path + "real/"
fpp_fake_path = fpp_frames_path + "fake/"

fpp_img_paths = [fpp_real_path + img_path for img_path in os.listdir(fpp_real_path)] + [fpp_fake_path + img_path for img_path in os.listdir(fpp_fake_path)]
fpp_img_labels = [0] * len(os.listdir(fpp_real_path)) + [1] * len(os.listdir(fpp_fake_path))

# celeb-df dataset
cdf_frames_path = "../data/celebdf-frame/" # change to appropriate directory. only using test set here.
cdf_real_path = cdf_frames_path + "real/"
cdf_fake_path = cdf_frames_path + "fake/"

cdf_img_paths = [cdf_real_path + img_path for img_path in os.listdir(cdf_real_path)] + [cdf_fake_path + img_path for img_path in os.listdir(cdf_fake_path)]
cdf_img_labels = [0] * len(os.listdir(cdf_real_path)) + [1] * len(os.listdir(cdf_fake_path))

# datasets and loaders with augmentation
fpp_wtrf_train_set, fpp_wtrf_val_set = random_split(DeepfakeImageDataset(fpp_img_paths, fpp_img_labels, transform=aug_transform), [0.8, 0.2], gen)
fpp_wtrf_train_loader = DataLoader(fpp_wtrf_train_set, 64, shuffle=True)
fpp_wtrf_val_loader = DataLoader(fpp_wtrf_val_set, 64, shuffle=True)

# datasets and loaders without augmentation
fpp_train_set, fpp_val_set = random_split(DeepfakeImageDataset(fpp_img_paths, fpp_img_labels, transform=base_transform), [0.8, 0.2], gen)
fpp_train_loader = DataLoader(fpp_train_set, 64, shuffle=True)
fpp_val_loader = DataLoader(fpp_val_set, 64, shuffle=True)

# celeb-df validation dataset
cdf_dataset = DeepfakeImageDataset(cdf_img_paths, cdf_img_labels, transform=base_transform)
cdf_dataloader = DataLoader(cdf_dataset, 64, shuffle=True)

In [ ]:
# Keep track of experiment results to print standard tables later
experiment_results = {}

# ---------------------------------------------------------
# 1. Experiment 1 - Baseline (RGB only, No Augmentation)
# Expected Result: Massive domain shift drop (large gap)
# ---------------------------------------------------------
model_rgb = HybridDeepfakeDetector(mode='rgb').to(device)
stats = train_model(model_rgb, fpp_train_loader, fpp_val_loader, 10, device)
gap_rgb = evaluate_generalization_gap(model_rgb, fpp_val_loader, cdf_dataloader, 'rgb', device)
experiment_results['RGB Only'] = gap_rgb, stats

# ---------------------------------------------------------
# 2. Experiment 2 - Augmentation (RGB only)
# Expected Result: Small improvement to generalization gap over pure RGB
# ---------------------------------------------------------
model_aug = HybridDeepfakeDetector(mode='rgb').to(device)
stats = train_model(model_aug, fpp_wtrf_train_loader, fpp_wtrf_val_loader, 10, device)
gap_aug = evaluate_generalization_gap(model_aug, fpp_wtrf_val_loader, cdf_dataloader, 'rgb', device)
experiment_results['RGB + Augmentation'] = gap_aug, stats

# ---------------------------------------------------------
# 3. Experiment 3 - FFT Only (Frequency structure)
# Expected Result: Generalization gap is moderate compared to RGB only
# ---------------------------------------------------------
model_fft = HybridDeepfakeDetector(mode='fft').to(device)
stats = train_model(model_fft, fpp_train_loader, fpp_val_loader, 10, device)
gap_fft = evaluate_generalization_gap(model_fft, fpp_val_loader, cdf_dataloader, 'fft', device)
experiment_results['FFT Only'] = gap_fft, stats

# ---------------------------------------------------------
# 4. Experiment 4 - Hybrid/Combined (RGB + FFT + Augmentation)
# Expected Result: Smallest Generalization drop due to robust frequency/spatial combos
# ---------------------------------------------------------
model_hyb = HybridDeepfakeDetector(mode='hybrid').to(device)
stats = train_model(model_hyb, fpp_wtrf_train_loader, fpp_wtrf_val_loader, 10, device)
gap_hybrid = evaluate_generalization_gap(model_hyb, fpp_wtrf_val_loader, cdf_dataloader, 'hybrid', device)
experiment_results['Hybrid (RGB + FFT)'] = gap_hybrid, stats

## 6. Analyze the Results (Step 8)
Record your metrics systematically to calculate the **Generalization Gap**:
`Generalization Gap = In-domain (FF++) Accuracy - Cross-domain (Celeb-DF) Accuracy`.

### Expected Behavior to Note in your Report:
1. **Model: RGB Only | Gap: Large Drop (e.g. 95% -> 40%)**
    *   *Why baseline fails (Exp 1):* The model heavily over-learns dataset-specific attributes (specific lighting, face blending borders, or artifacts typical to FF++ like Face2Face/FaceSwap signatures). When tested on Celeb-DF, it completely drops in accuracy.
2. **Model: RGB + Augmentation | Gap: Moderate Reduction**
    *   *Why augmentation helps (Exp 2):* Introducing generic noise, compression, and blur removes the "visual shortcuts" the baseline relied on, forcing the network to look for broader patterns. 
3. **Model: FFT Only | Gap: Moderate Drop**
    *   *Why FFT helps (Exp 3):* Deepfakes generated by GANs/blending leave distinct checkerboard artifacts and high-frequency noise that the human eye (and RGB layers) miss. Processing the FFT forces the network to look for spectral inconsistencies, which are more universal across generative methods.
4. **Model: Hybrid | Gap: Smallest Drop (Your Contribution)**
    *   *Why this works (Exp 4):* By processing RGB and FFT in **parallel branches** before concatenating, the model learns both facial structural anomalies AND frequency anomalies concurrently. This combination provides the most robust generalization out-of-domain.

In [7]:
experiment_results = {}

model_rgb = HybridDeepfakeDetector(mode='rgb').to(device)
state_dict = torch.load('../models/model_rgb_20260416_091501_9', weights_only=True)
model_rgb.load_state_dict(state_dict)
gap_rgb = evaluate_generalization_gap(model_rgb, fpp_val_loader, cdf_dataloader, 'rgb', device)
experiment_results['RGB Only'] = gap_rgb

model_aug = HybridDeepfakeDetector(mode='rgb').to(device)
state_dict = torch.load('../models/model_rgb_20260416_093803_9', weights_only=True)
model_aug.load_state_dict(state_dict)
gap_aug = evaluate_generalization_gap(model_aug, fpp_wtrf_val_loader, cdf_dataloader, 'rgb', device)
experiment_results['RGB + Augmentation'] = gap_aug

model_fft = HybridDeepfakeDetector(mode='fft').to(device)
state_dict = torch.load('../models/model_fft_20260416_095640_8', weights_only=True)
model_fft.load_state_dict(state_dict)
gap_fft = evaluate_generalization_gap(model_fft, fpp_val_loader, cdf_dataloader, 'fft', device)
experiment_results['FFT Only'] = gap_fft

model_hyb = HybridDeepfakeDetector(mode='hybrid').to(device)
state_dict = torch.load('../models/model_hybrid_20260416_102842_7', weights_only=True)
model_hyb.load_state_dict(state_dict)
gap_hyb = evaluate_generalization_gap(model_hyb, fpp_wtrf_val_loader, cdf_dataloader, 'hybrid', device)
experiment_results['Hybrid (RGB + FFT)'] = gap_hyb

experiment_results

--- Model Setup [RGB] ---
In-domain (FF++)      : Acc=98.55%, F1=0.989, AUC=0.999
Cross-domain (CelebDF): Acc=61.52%, F1=0.536, AUC=0.686
GENERALIZATION GAP: 37.02%

--- Model Setup [RGB] ---
In-domain (FF++)      : Acc=95.25%, F1=0.964, AUC=0.990
Cross-domain (CelebDF): Acc=64.28%, F1=0.610, AUC=0.709
GENERALIZATION GAP: 30.98%

--- Model Setup [FFT] ---
In-domain (FF++)      : Acc=81.78%, F1=0.867, AUC=0.898
Cross-domain (CelebDF): Acc=59.94%, F1=0.670, AUC=0.661
GENERALIZATION GAP: 21.84%

--- Model Setup [HYBRID] ---
In-domain (FF++)      : Acc=93.30%, F1=0.948, AUC=0.982
Cross-domain (CelebDF): Acc=69.47%, F1=0.675, AUC=0.774
GENERALIZATION GAP: 23.83%



{'RGB Only': 0.3702017611026034,
 'RGB + Augmentation': 0.3097767993874425,
 'FFT Only': 0.21838916539050535,
 'Hybrid (RGB + FFT)': 0.23825153139356814}